# Занятие 8. Финальная интеграция и агентная проверка пакета

**Цель практики:** собрать минимальный финальный пакет проекта в Colab, проверить его чек-листом и получить список задач перед публикацией/летней школой.

Работает в бесплатном Colab на CPU.

In [ ]:
import os, re, json, textwrap, math, statistics, random, io, time
from pathlib import Path
import pandas as pd
import numpy as np
import requests

DATA_DIR = Path('/content/lowres_lab')
DATA_DIR.mkdir(exist_ok=True)

def show_df(df, n=10):
    display(df.head(n))

def save_artifact(name, obj):
    path = DATA_DIR / name
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(path, index=False)
    else:
        path.write_text(str(obj), encoding='utf-8')
    print('saved:', path)

## 1. Создаем мини-пакет проекта

In [ ]:
PROJECT = DATA_DIR / 'final_package'
PROJECT.mkdir(exist_ok=True)

files = {
    'README.md': '''# Mini low-resource language project

Goal: create a tiny documented corpus prototype.
Language: Udmurt / replace with your language.
Status: classroom prototype, not production.
''',
    'DATA_CARD.md': '''# Data card

Sources: Wikipedia API / Wikimedia Commons / replace with your sources.
License: check source pages before redistribution.
Known limitations: small sample, not representative, needs human review.
''',
    'EVAL_REPORT.md': '''# Evaluation report

Metric: manual inspection + simple script diagnostics.
Result: baseline works partially.
Errors: needs language expert review.
''',
    'sources.csv': 'title,url,type,license_or_access,notes\nExample,https://example.org,web,unknown,replace me\n',
}
for name, content in files.items():
    (PROJECT / name).write_text(content, encoding='utf-8')

print('created files:')
for p in sorted(PROJECT.iterdir()):
    print('-', p.name)

## 2. Проверяем структуру пакета

In [ ]:
required = ['README.md', 'DATA_CARD.md', 'EVAL_REPORT.md', 'sources.csv']
checks = []
for name in required:
    p = PROJECT / name
    checks.append({
        'check': f'{name} exists',
        'ok': p.exists(),
        'details': str(p),
    })
    if p.exists():
        txt = p.read_text(encoding='utf-8')
        checks.append({
            'check': f'{name} is not empty',
            'ok': len(txt.strip()) > 40,
            'details': f'{len(txt)} chars',
        })

sources = pd.read_csv(PROJECT / 'sources.csv')
for col in ['title', 'url', 'type', 'license_or_access', 'notes']:
    checks.append({'check': f'sources.csv has column {col}', 'ok': col in sources.columns, 'details': ''})

check_df = pd.DataFrame(checks)
show_df(check_df, 30)
save_artifact('lesson08_package_checks.csv', check_df)

## 3. Агентный review без LLM: правила и задачи

In [ ]:
tasks = []

if not check_df['ok'].all():
    tasks.append('исправить отсутствующие или пустые обязательные файлы')

if 'unknown' in sources['license_or_access'].fillna('').str.lower().to_string():
    tasks.append('уточнить лицензии в sources.csv')

readme = (PROJECT / 'README.md').read_text(encoding='utf-8').lower()
if 'replace' in readme:
    tasks.append('заменить placeholder-описания в README.md')

data_card = (PROJECT / 'DATA_CARD.md').read_text(encoding='utf-8').lower()
if 'human review' in data_card or 'needs' in data_card:
    tasks.append('запланировать ручную проверку носителем/экспертом')

report = {
    'package_ready': len(tasks) == 0,
    'tasks_before_publication': tasks,
    'summer_school_angle': [
        'какой артефакт можно показать участникам',
        'какие данные нужно дособрать',
        'какие роли нужны в команде',
        'какие риски нельзя автоматизировать',
    ],
}
print(json.dumps(report, ensure_ascii=False, indent=2))
save_artifact('lesson08_agent_review.json', json.dumps(report, ensure_ascii=False, indent=2))

## 4. Финальная таблица для защиты

In [ ]:
def status(ok):
    return 'готово' if ok else 'нужно доработать'

def exists(name):
    return (PROJECT / name).exists()

def nonempty(name):
    p = PROJECT / name
    return p.exists() and len(p.read_text(encoding='utf-8').strip()) > 40

defense = pd.DataFrame([
    {'artifact': 'README', 'status': status(nonempty('README.md')), 'next_step': 'уточнить цель и пользователей'},
    {'artifact': 'sources.csv', 'status': status(exists('sources.csv')), 'next_step': 'добавить реальные лицензии'},
    {'artifact': 'DATA_CARD', 'status': status(nonempty('DATA_CARD.md')), 'next_step': 'описать ограничения'},
    {'artifact': 'EVAL_REPORT', 'status': status(nonempty('EVAL_REPORT.md')), 'next_step': 'добавить примеры ошибок'},
])
show_df(defense)
save_artifact('lesson08_defense_table.csv', defense)

## Вопросы для отчёта

1. Что в пакете уже можно показать внешнему человеку?
2. Что нельзя публиковать без дополнительной проверки?
3. Какие задачи переходят в летнюю школу?
4. Какие проверки стоит автоматизировать агентом?